# Embeddings extracted from STraTs pretrained model

In [ ]:
# no finetuning
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config.constants import PROJECT_ROOT


In [ ]:


EMB_DIR    = Path(PROJECT_ROOT) / "saved_data" / "embeddings"
STRATS_DIR = EMB_DIR / "strats_pretrained"
EIC_DIR    = EMB_DIR / "eic_bert_embeddings_2085cohorts"


def fold_table(cohort, fold=0, seed=42):
    # hadm_id, our split and label for one cohort
    f = Path(PROJECT_ROOT) / "saved_data" / "folds" / cohort / f"seed_{seed}_firstadm" / f"fold_{fold}.pkl"
    train, val, test = pickle.load(open(f, "rb"))
    split = pd.concat([pd.DataFrame({"hadm_id": a[:, 1], "split": s})
                       for a, s in [(train, "train"), (val, "val"), (test, "test")]])
    labels = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "cohorts" / "DTB" / "new" / f"{cohort}.csv.gz",
                         usecols=["hadm_id", "label"])
    return split.merge(labels, on="hadm_id")


def load_strats(cohort, fold=0):
    d = STRATS_DIR / cohort / f"fold_{fold}"
    return pd.concat([pd.read_parquet(d / f"strats_embeddings_{s}.parquet")
                      for s in ("train", "val", "test")], ignore_index=True)
    
    


def load_eic(cohort, fold=0):
    ft = fold_table(cohort, fold)
    ids = ft.hadm_id.tolist()
    # read only this cohort's admissions, the full files are large
    eic = pd.concat([pd.read_parquet(EIC_DIR / f"eic_bert_embeddings_{s}.parquet",
                                     filters=[("hadm_id", "in", ids)])
                     for s in ("train", "tuning", "test")], ignore_index=True)
    # replace the BERT split with our fold split
    return eic.drop(columns="split").merge(ft, on="hadm_id")


def to_xy(df, split, col="embedding"):
    d = df[df.split == split]
    return np.stack(d[col].values), d.label.to_numpy(), d.hadm_id.to_numpy()


In [ ]:
cohort = "A08-K52"
strats = load_strats(cohort)
eic    = load_eic(cohort)

# in strats we have missing admissions because of no data (labs)
common = np.intersect1d(strats.hadm_id, eic.hadm_id)
strats = strats[strats.hadm_id.isin(common)].copy()
eic    = eic[eic.hadm_id.isin(common)]

strats["embedding"] = [np.concatenate([a, b]) for a, b in zip(strats.embedding, strats.embedding_demo)]

X_s, y_s, ids_s = to_xy(strats, "train")   # (n, 192)
X_e, y_e, ids_e = to_xy(eic, "train")      # (n, 768)



# Train on embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def probe(df, Cs=(1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100)
):
    X_tr, y_tr, _ = to_xy(df, "train")
    X_va, y_va, _ = to_xy(df, "val")
    X_te, y_te, _ = to_xy(df, "test")

    # pick C on val
    best_C, best_auc = None, -1
    for C in Cs:
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(C=C, class_weight="balanced", max_iter=2000))
        clf.fit(X_tr, y_tr)
        auc = roc_auc_score(y_va, clf.predict_proba(X_va)[:, 1])
        if auc > best_auc:
            best_C, best_auc = C, auc

    # refit on train+val, evaluate on test
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=best_C, class_weight="balanced", max_iter=2000))
    clf.fit(np.vstack([X_tr, X_va]), np.concatenate([y_tr, y_va]))
    p = clf.predict_proba(X_te)[:, 1]
    return {"C": best_C, "auroc": roc_auc_score(y_te, p), "auprc": average_precision_score(y_te, p),
            "target_rate": y_te.mean()}

pd.DataFrame({"STraTS (192)": probe(strats), "EIC-BERT (768)": probe(eic)}).T.round(3)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def probe_rf(df, depths=(5, 10, 30, None)):
    X_tr, y_tr, _ = to_xy(df, "train")
    X_va, y_va, _ = to_xy(df, "val")
    X_te, y_te, _ = to_xy(df, "test")

    # pick max_depth on val
    best_d, best_auc = None, -1
    for d in depths:
        clf = RandomForestClassifier(n_estimators=100, max_depth=d, class_weight="balanced",
                                     n_jobs=-1, random_state=42)
        clf.fit(X_tr, y_tr)
        auc = roc_auc_score(y_va, clf.predict_proba(X_va)[:, 1])
        if auc > best_auc:
            best_d, best_auc = d, auc

    # refit on train+val, evaluate on test
    clf = RandomForestClassifier(n_estimators=100, max_depth=best_d, class_weight="balanced",
                                 n_jobs=-1, random_state=42)
    clf.fit(np.vstack([X_tr, X_va]), np.concatenate([y_tr, y_va]))
    p = clf.predict_proba(X_te)[:, 1]
    return {"max_depth": best_d, "auroc": roc_auc_score(y_te, p), "auprc": average_precision_score(y_te, p),
            "target_rate": y_te.mean()}

pd.DataFrame({"STraTS (192) RF": probe_rf(strats), "EIC-BERT (768) RF": probe_rf(eic)}).T.round(3)


# Shortcut Analysis


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BIAS_DIR = Path(PROJECT_ROOT) / "saved_data" / "bias_cv"
ATTRS    = ["n_events", "n_labs", "age", "gender"]

rows = []
for cohort_dir in BIAS_DIR.iterdir():
    for attr in ATTRS:
        f = cohort_dir / f"{attr}_bias_mqc.tsv"
        if f.exists():
            t = pd.read_csv(f, sep="\t", comment="#")
            rows.append(t.assign(cohort=cohort_dir.name, attribute=attr))
bias = (pd.concat(rows, ignore_index=True)
          .rename(columns={"Split": "split", "NMI(A;Y)": "utility",
                           "Detectability (Spearman ρ)": "detectability"}))
print(bias.cohort.nunique(), "cohorts")

#SPLIT_STYLE = {"all": ("#2a78d6", "o")}
SPLIT_STYLE = {"train": ("#2a78d6", "o"), "val": ("#1baf7a", "^"), "test": ("#eb6834", "s")}

fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
axes = axes.ravel()
for ax, attr in zip(axes, ATTRS):
    d = bias[bias.attribute == attr]
    for split, (c, m) in SPLIT_STYLE.items():
        s = d[d.split == split]
        ax.scatter(s.detectability, s.utility, color=c, marker=m, s=20, alpha=0.3, lw=0, label=split)
        # median over cohorts
        ax.scatter(s.detectability.median(), s.utility.median(), color=c, marker=m, s=80,
                   edgecolor="black", lw=1.5, zorder=3)
    ax.set_title(attr, fontsize=16)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=12)
    ax.grid(color="#e5e4e0", lw=0.8)
for ax in axes[2:]:
    ax.set_xlabel("Detectability", fontsize=16)
for ax in axes[::2]:
    ax.set_ylabel("Utility: NMI(a; y)", fontsize=16)
axes[1].legend(frameon=False, fontsize=16, markerscale=2.5)
fig.suptitle(f"STraTS pretrained embeddings: {bias.cohort.nunique()} cohorts", fontsize=18)
fig.tight_layout()

In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm

ATTRS = ["n_events", "n_labs", "adm_missing", "itembin_missing", "age", "gender"]

bias = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "bias_results" / "bias_cv_pooled_perm.tsv", sep="\t")
          .query("attribute in @ATTRS")
          .rename(columns={"Split": "split", "NMI(A;Y)": "utility", "Detectability (Spearman ρ)": "detectability",
                           "NMI p-value": "p_perm", "NMI z-score": "z"}))

# p from permutation z-score (100 perms too few for FDR over 1002 cohorts)
bias["p"] = norm.sf(bias.z).clip(0, 1)
bias["p"] = bias["p"].fillna(1.0)

# FDR per attribute, 1002 cohorts tested each
bias["q"] = bias.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
bias["significant"] = bias.q <= 0.05

fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True)
axes = axes.ravel()
for ax, attr in zip(axes, ATTRS):
    d = bias[bias.attribute == attr]
    ns, sg = d[~d.significant], d[d.significant]
    ax.scatter(ns.detectability, ns.utility, color="#c8c7c3", s=15, alpha=0.9, lw=0, label="not significant")
    ax.scatter(sg.detectability, sg.utility, color="#d6453d", s=25, alpha=0.8, lw=0, label="significant (q < 0.05)")
    ax.set_title(f"{attr}  ({len(sg)}/{len(d)} significant)", fontsize=14)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=11)
    ax.grid(color="#e5e4e0", lw=0.8)
for ax in axes[3:]:
    ax.set_xlabel("Detectability (CV Spearman ρ)", fontsize=13)
for ax in axes[::3]:
    ax.set_ylabel("Utility: NMI(a; y)", fontsize=13)
axes[2].legend(frameon=False, fontsize=11)
fig.suptitle(f"STraTS pretrained embeddings: {bias.cohort.nunique()} cohorts (pooled)", fontsize=15)
fig.tight_layout()


In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
import matplotlib.pyplot as plt

bias_bert = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "bias_results" / "bias_bert_pooled_perm.tsv", sep="\t")
               .rename(columns={"Split": "split", "NMI(A;Y)": "utility", "Detectability (Spearman ρ)": "detectability",
                                "NMI p-value": "p_perm", "NMI z-score": "z"}))
ATTRS_BERT = sorted(bias_bert.attribute.unique())

bias_bert["p"] = norm.sf(bias_bert.z).clip(0, 1)
bias_bert["p"] = bias_bert["p"].fillna(1.0)
bias_bert["q"] = bias_bert.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
bias_bert["significant"] = bias_bert.q <= 0.05

fig, axes = plt.subplots(5, 5, figsize=(20, 18), sharex=True, sharey=True)
axes = axes.ravel()
for ax, attr in zip(axes, ATTRS_BERT):
    d = bias_bert[bias_bert.attribute == attr]
    ns, sg = d[~d.significant], d[d.significant]
    ax.scatter(ns.detectability, ns.utility, color="#c8c7c3", s=15, alpha=0.9, lw=0, label="not significant")
    ax.scatter(sg.detectability, sg.utility, color="#d6453d", s=25, alpha=0.8, lw=0, label="significant (q < 0.05)")
    ax.set_title(f"{attr}  ({len(sg)}/{len(d)})", fontsize=14)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=11)
    ax.grid(color="#e5e4e0", lw=0.8)
for ax in axes[20:]:
    ax.set_xlabel("Detectability", fontsize=16)
for ax in axes[::5]:
    ax.set_ylabel("Utility: NMI(a; y)", fontsize=16)
axes[4].legend(frameon=False, fontsize=11)
fig.suptitle(f"BERT embeddings: {bias_bert.cohort.nunique()} cohorts (pooled)", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.98])


In [ ]:
MODEL = "strats"    
ATTR  = "gender"

df = {"strats": bias, "bert": bias_bert}[MODEL]
chars = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "characteristics" / "cohort_characteristics_14d_agg24_top100_train+val.csv")


In [ ]:
#bert attributes
['n_lab_events', 'n_tokens_model_input', 'n_tokens_untruncated',
       'n_medication_events', 'n_events_window', 'n_events_admission',
       'los_days', 'days_since_prev_discharge', 'n_diagnosis_events',
       'n_transfer_events', 'n_ed_events', 'window_span_days',
       'n_transfers', 'n_prior_admissions', 'n_procedure_events',
       'n_events_pre_admission', 'gender', 'truncated_at_context',
       'age_at_admission_years', 'icu_any', 'race_group=BLACK',
       'n_icu_stays', 'race_group=WHITE', 'has_medication', 'approx_year']

#STraTs attributes
['gender', 'n_events', 'n_labs', 'age', 'itembin_missing',
       'adm_missing']

In [ ]:
target_att_bert = 'n_lab_events'
target_att_strats = 'n_events'

In [ ]:
cohorts_bert = bias_bert.loc[(bias_bert.attribute == target_att_bert) & bias_bert.significant, "cohort"].tolist()
print(len(cohorts_bert))
cohorts_strats = bias.loc[(bias.attribute == target_att_strats) & bias.significant, "cohort"].tolist()
print(len(cohorts_strats))

common_cohorts = set(cohorts_bert) & set(cohorts_strats)
print(len(common_cohorts))


In [ ]:
chars = pd.read_csv(Path(PROJECT_ROOT) / "saved_data/characteristics/cohort_characteristics_14d_agg24_top100_train+val.csv")

# BERT lab events per cohort, first admission per patient
lab = pd.read_csv(Path(PROJECT_ROOT) / "saved_data/embeddings/eic_bert_embeddings_2085cohorts/shortcut_candidates_K1000.csv",
                  usecols=["hadm_id", "n_lab_events"]).drop_duplicates("hadm_id").set_index("hadm_id").n_lab_events

rows = []
for c in chars.cohort_name:
    coh = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data/cohorts/DTB/new" / f"{c}.csv.gz",
                       usecols=["subject_id", "hadm_id", "admittime", "label"])
             .sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first"))
    d = coh.join(lab, on="hadm_id").dropna(subset=["n_lab_events"])
    rows.append({"cohort_name": c,
                 "bert_lab_mean": d.n_lab_events.mean(),
                 "bert_lab_mean_pos": d.loc[d.label == 1, "n_lab_events"].mean(),
                 "bert_lab_mean_neg": d.loc[d.label == 0, "n_lab_events"].mean()})
chars = chars.merge(pd.DataFrame(rows), on="cohort_name", how="left")

sb, ss = set(cohorts_bert), set(cohorts_strats)

LAB = {"mean": ("n_events_mean", "bert_lab_mean"),
       "mean_pos": ("n_events_mean_pos", "bert_lab_mean_pos"),
       "mean_neg": ("n_events_mean_neg", "bert_lab_mean_neg")}

rng = np.random.default_rng(0)
COLORS = ["#d6453d", "#c8c7c3", "#d6453d", "#c8c7c3"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, (s_col, b_col)) in zip(axes, LAB.items()):
    groups = [chars.loc[chars.cohort_name.isin(ss), s_col].dropna(),
              chars.loc[~chars.cohort_name.isin(ss), s_col].dropna(),
              chars.loc[chars.cohort_name.isin(sb), b_col].dropna(),
              chars.loc[~chars.cohort_name.isin(sb), b_col].dropna()]
    for i, (vals, color) in enumerate(zip(groups, COLORS)):
        ax.scatter(i + rng.uniform(-0.18, 0.18, len(vals)), vals, color=color, s=10, alpha=0.6, lw=0)
    ax.boxplot(groups, positions=range(4), widths=0.5, showfliers=False,
               medianprops=dict(color="black", lw=1.5), boxprops=dict(lw=1), whiskerprops=dict(lw=1), capprops=dict(lw=1))
    ax.set_xticks(range(4), ["STraTS sig", "STraTS not sig", "BERT sig", "BERT not sig"], rotation=30)
    ax.set_title(f"lab events {name}", fontsize=12)
    ax.grid(axis="y", color="#e5e4e0", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()


In [ ]:
MODEL = "strats"      # "strats" or "bert"
ATTR  = "gender"
COLS  = ["n"]   # any columns of chars

df  = {"strats": bias, "bert": bias_bert}[MODEL]
sig = set(df.loc[(df.attribute == ATTR) & df.significant, "cohort"])
is_sig = chars.cohort_name.isin(sig)

ncols = min(len(COLS), 4)
nrows = -(-len(COLS) // ncols)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)
for ax, c in zip(axes.ravel(), COLS):
    groups = [chars.loc[~is_sig, c].dropna(), chars.loc[is_sig, c].dropna()]
    for i, (vals, color) in enumerate(zip(groups, ["#c8c7c3", "#d6453d"])):
        ax.scatter(i + rng.uniform(-0.18, 0.18, len(vals)), vals, color=color, s=10, alpha=0.6, lw=0)
    ax.boxplot(groups, positions=[0, 1], widths=0.5, showfliers=False,
               medianprops=dict(color="black", lw=1.5), boxprops=dict(lw=1), whiskerprops=dict(lw=1), capprops=dict(lw=1))
    ax.set_xticks([0, 1], ["not sig", "sig"])
    ax.set_title(c, fontsize=12)
    ax.grid(axis="y", color="#e5e4e0", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    if c == "n":
        ax.set_yscale("log")
for ax in axes.ravel()[len(COLS):]:
    ax.set_visible(False)
fig.suptitle(f"{MODEL}: {ATTR}  ({is_sig.sum()} sig / {(~is_sig).sum()} not sig)", fontsize=14)
fig.tight_layout()


# Linear Probe

In [ ]:
coh_all = pd.concat([
    pd.read_csv(Path(PROJECT_ROOT) / "saved_data/cohorts/DTB/new" / f"{c}.csv.gz",
                usecols=["subject_id", "hadm_id", "admittime", "label"])
      .sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first")
      .assign(cohort=c)[["cohort", "hadm_id", "label"]]
    for c in chars.cohort_name], ignore_index=True)

attrs = pd.read_csv(Path(PROJECT_ROOT) / "saved_data/embeddings/eic_bert_embeddings_2085cohorts/shortcut_candidates_K1000.csv") \
          .drop_duplicates("hadm_id").set_index("hadm_id")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

probe = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "probe_lr_all_cohorts.csv")
COLORS = {"STraTS": "#2a78d6", "EIC-BERT": "#eb6834"}
metric = "auroc"   # or "auprc"

p = probe.pivot(index="cohort", columns="model", values=metric).dropna()
d = p["EIC-BERT"] - p["STraTS"]
print(f"BERT better in {(d > 0).sum()}/{len(d)} cohorts, mean diff {d.mean():+.3f}, {wilcoxon(p['EIC-BERT'], p['STraTS'])}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# 1. paired per cohort
lo, hi = p.min().min() - 0.02, p.max().max() + 0.02
ax1.plot([lo, hi], [lo, hi], color="#8a8985", lw=1, ls="--", zorder=1)
ax1.scatter(p["STraTS"], p["EIC-BERT"], s=14, alpha=0.5, color="#5c5b56", lw=0, zorder=2)
ax1.set(xlim=(lo, hi), ylim=(lo, hi), aspect="equal",
        xlabel=f"STraTS {metric.upper()}", ylabel=f"EIC-BERT {metric.upper()}",
        title=f"Per cohort: BERT above the line in {(d > 0).mean():.0%}")
ax1.grid(color="#e5e4e0", lw=0.8)

# 2. distribution per model
models = ["STraTS", "EIC-BERT"]
rng = np.random.default_rng(42)
for i, m in enumerate(models, start=1):
    ax2.scatter(i + rng.uniform(-0.15, 0.15, len(p)), p[m], s=8, alpha=0.15, color=COLORS[m], lw=0, zorder=1)

bp = ax2.boxplot([p[m] for m in models], widths=0.5, showfliers=False, patch_artist=True, zorder=2,
                 medianprops=dict(color="#1a1a19", lw=2))
for patch, m in zip(bp["boxes"], models):
    patch.set(facecolor=COLORS[m], alpha=0.7, edgecolor="none")
ax2.set_xticks([1, 2], [f"{m}\n{p[m].mean():.3f} ± {p[m].std():.3f}" for m in models])
ax2.set(ylabel=metric.upper(), title="Linear probe on frozen embeddings")
ax2.grid(axis="y", color="#e5e4e0", lw=0.8)

fig.suptitle(f"Linear probe, {len(p)} cohorts", fontsize=14)
fig.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# data
probe = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "probe_lr_all_cohorts.csv")
p = probe.pivot(index="cohort", columns="model", values="auroc").dropna()
cc = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "characteristics" / "characteristics_14d_agg24.csv").set_index("cohort_name")
df = cc.join((p["EIC-BERT"] - p["STraTS"]).rename("gap"), how="inner")

CHARS = ["n", "target_rate", "n_events_mean", "itembin_missing_mean", "adm_missing_mean",
         "cohens_d_adj_z", "h_diff", "age", "female_rate", "death_rate", "RR", "W"]
LOG = {"n", "RR", "W"}   # skewed, shown on a log axis

df["winner"] = np.where(df["gap"] < 0, "STraTS better", "BERT better")
GROUPS = ["STraTS better", "BERT better"]
COLORS = {"STraTS better": "#2a78d6", "BERT better": "#eb6834"}
rng = np.random.default_rng(42)

fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for ax, c in zip(axes.ravel(), CHARS):
    vals = [df.loc[df.winner == g, c].dropna() for g in GROUPS]
    for i, (g, v) in enumerate(zip(GROUPS, vals), start=1):
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=6, alpha=0.15, color=COLORS[g], lw=0, zorder=1)
    ax.boxplot(vals, widths=0.5, showfliers=False, zorder=2,
               medianprops=dict(color="#1a1a19", lw=2))
    pval = mannwhitneyu(*vals).pvalue
    ax.set_xticks([1, 2], [f"{g}\n(n={len(v)})" for g, v in zip(GROUPS, vals)], fontsize=9)
    if c in LOG:
        ax.set_yscale("log")
    ax.set_title(f"{c}   p = {pval:.3f}{' *' if pval < 0.05 else ''}", fontsize=12)
    ax.grid(axis="y", color="#e5e4e0", lw=0.8)
fig.suptitle("Characteristics of cohorts where STraTS vs BERT is better", fontsize=14)
fig.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

R = Path(PROJECT_ROOT) / "saved_data" / "results"
metric = "auroc"   # or "auprc"

probe = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "probe_lr_all_cohorts.csv")
lp = probe[probe.model == "STraTS"].set_index("cohort")[metric].rename("Linear probe")

ft = {}
for f in R.glob("*/strats/finetune_strats_14d/fold_0/results_final.csv"):
    r = pd.read_csv(f)
    ft[f.parts[-5]] = r.loc[r.split == "test", metric].iloc[0]
ft = pd.Series(ft, name="Fine-tuned")

p = pd.concat([lp, ft], axis=1).dropna()
d = p["Fine-tuned"] - p["Linear probe"]
print(f"{len(p)} cohorts | fine-tuned better in {(d > 0).sum()}, mean diff {d.mean():+.3f}, {wilcoxon(p['Fine-tuned'], p['Linear probe'])}")

COLORS = {"Linear probe": "#2a78d6", "Fine-tuned": "#468872"}
models = ["Linear probe", "Fine-tuned"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# 1. paired per cohort
lo, hi = p.min().min() - 0.02, p.max().max() + 0.02
ax1.plot([lo, hi], [lo, hi], color="#8a8985", lw=1, ls="--", zorder=1)
ax1.scatter(p["Linear probe"], p["Fine-tuned"], s=14, alpha=0.5, color="#5c5b56", lw=0, zorder=2)
ax1.set(xlim=(lo, hi), ylim=(lo, hi), aspect="equal",
        xlabel=f"Linear probe {metric.upper()}", ylabel=f"Fine-tuned {metric.upper()}",
        title=f"Per cohort: fine-tuned above the line in {(d > 0).mean():.0%}")
ax1.grid(color="#e5e4e0", lw=0.8)

# 2. distribution with dots
rng = np.random.default_rng(42)
for i, m in enumerate(models, start=1):
    ax2.scatter(i + rng.uniform(-0.15, 0.15, len(p)), p[m], s=8, alpha=0.15, color=COLORS[m], lw=0, zorder=1)
bp = ax2.boxplot([p[m] for m in models], widths=0.5, showfliers=False, patch_artist=True, zorder=2,
                 medianprops=dict(color="#1a1a19", lw=2))
for patch, m in zip(bp["boxes"], models):
    patch.set(facecolor=COLORS[m], alpha=0.7, edgecolor="none")
ax2.set_xticks([1, 2], [f"{m}\n{p[m].mean():.3f} ± {p[m].std():.3f}" for m in models])
ax2.set(ylabel=metric.upper(), title="STraTS: frozen embeddings vs fine-tuning")
ax2.grid(axis="y", color="#e5e4e0", lw=0.8)

fig.suptitle(f"STraTS, {len(p)} cohorts", fontsize=14)
fig.tight_layout()


In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
import matplotlib.pyplot as plt


bias = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "bias_results" / "bias_bert_pooled_perm.tsv", sep="\t")
          .rename(columns={"NMI(A;Y)": "utility", "Detectability (Spearman ρ)": "detectability", "NMI z-score": "z"}))
bias["p"] = norm.sf(bias.z).clip(0, 1)
bias["p"] = bias["p"].fillna(1.0)

bias["q"] = bias.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
bias["significant"] = bias.q <= 0.05

# cohort size and positive rate
sizes = []
for c in bias.cohort.unique():
    y = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "cohorts" / "DTB" / "new" / f"{c}.csv.gz", usecols=["label"]).label
    sizes.append({"cohort": c, "n": len(y), "pos_rate": y.mean()})
bias = bias.merge(pd.DataFrame(sizes), on="cohort")

# per attribute: does n drive detectability and significance?
summary = bias.groupby("attribute").apply(lambda d: pd.Series({
    "rho_det_n":     spearmanr(d.detectability, d.n, nan_policy="omit")[0],
    "rho_util_n":    spearmanr(d.utility, d.n, nan_policy="omit")[0],
    "median_n_sig":  d[d.significant].n.median(),
    "median_n_ns":   d[~d.significant].n.median(),
}))
print(summary.round(2))

# same scatter, colored by log10(n)
attrs = sorted(bias.attribute.unique())
fig, axes = plt.subplots(5, 5, figsize=(20, 18), sharex=True, sharey=True)
for ax, attr in zip(axes.ravel(), attrs):
    d = bias[bias.attribute == attr]
    sc = ax.scatter(d.detectability, d.utility, c=np.log10(d.n), cmap="viridis", s=12, lw=0)
    ax.set_title(attr, fontsize=12)
    ax.set_xlim(0, 1)
fig.colorbar(sc, ax=axes, label="log10(cohort size)")


In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm, spearmanr
import matplotlib.pyplot as plt

EMB_DIR    = Path(PROJECT_ROOT) / "saved_data" / "embeddings"
STRATS_DIR = EMB_DIR / "strats_pretrained"


bias = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "bias_results" / "bias_cv_pooled_perm.tsv", sep="\t")
          .rename(columns={"NMI(A;Y)": "utility", "Detectability (Spearman ρ)": "detectability", "NMI z-score": "z"}))
bias["p"] = norm.sf(bias.z).clip(0, 1)
bias["p"] = bias["p"].fillna(1.0)
bias["q"] = bias.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
bias["significant"] = bias.q <= 0.05

# cohort size from the STraTS embedding files
sizes = []
for c in bias.cohort.unique():
    y = pd.concat([pd.read_parquet(STRATS_DIR / c / "fold_0" / f"strats_embeddings_{s}.parquet", columns=["label"])
                   for s in ("train", "val", "test")]).label
    sizes.append({"cohort": c, "n": len(y), "pos_rate": y.mean()})
sizes = pd.DataFrame(sizes)
print(sizes.describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
bias = bias.merge(sizes, on="cohort")

summary = bias.groupby("attribute").apply(lambda d: pd.Series({
    "rho_det_n":    spearmanr(d.detectability, d.n, nan_policy="omit")[0],
    "rho_util_n":   spearmanr(d.utility, d.n, nan_policy="omit")[0],
    "median_n_sig": d[d.significant].n.median(),
    "median_n_ns":  d[~d.significant].n.median(),
}))
print(summary.round(2))

attrs = ["n_events", "n_labs", "adm_missing", "itembin_missing", "age", "gender"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True)
for ax, attr in zip(axes.ravel(), attrs):
    d = bias[bias.attribute == attr]
    sc = ax.scatter(d.detectability, d.utility, c=np.log10(d.n), cmap="viridis", s=12, lw=0)
    ax.set_title(attr, fontsize=12)
    ax.set_xlim(0, 1)
fig.colorbar(sc, ax=axes, label="log10(cohort size)")


In [ ]:
big = sizes.nlargest(200, "n").cohort
b = bias[bias.cohort.isin(big)].copy()
# redo FDR over the 200 cohorts only
b["q"] = b.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
b["significant"] = b.q <= 0.05
print("n range:", sizes[sizes.cohort.isin(big)].n.min(), "-", sizes.n.max())

attrs = sorted(b.attribute.unique())
ncol = 3 if len(attrs) <= 6 else 5
fig, axes = plt.subplots(int(np.ceil(len(attrs) / ncol)), ncol, figsize=(4 * ncol, 3.3 * np.ceil(len(attrs) / ncol)),
                         sharex=True, sharey=True)
for ax, attr in zip(axes.ravel(), attrs):
    d = b[b.attribute == attr]
    ns, sg = d[~d.significant], d[d.significant]
    ax.scatter(ns.detectability, ns.utility, color="#c8c7c3", s=15, alpha=0.9, lw=0, label="not significant")
    ax.scatter(sg.detectability, sg.utility, color="#d6453d", s=25, alpha=0.8, lw=0, label="significant (q < 0.05)")
    ax.set_title(f"{attr}  ({len(sg)}/{len(d)})", fontsize=12)
    ax.set_xlim(0, 1)
    ax.grid(color="#e5e4e0", lw=0.8)
axes.ravel()[0].legend(frameon=False, fontsize=10)
fig.suptitle("200 largest cohorts", fontsize=14)
fig.tight_layout()


In [ ]:
import pickle
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
import matplotlib.pyplot as plt

bias = (pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "bias_results" / "bias_bert_pooled_perm.tsv", sep="\t")
          .rename(columns={"NMI(A;Y)": "utility", "Detectability (Spearman ρ)": "detectability", "NMI z-score": "z"}))
bias["p"] = norm.sf(bias.z).clip(0, 1)
bias["p"] = bias["p"].fillna(1.0)

# cohort size from the first-admission fold files
sizes = []
for c in bias.cohort.unique():
    tr, va, te = pickle.load(open(Path(PROJECT_ROOT) / "saved_data" / "folds" / c / "seed_42_firstadm" / "fold_0.pkl", "rb"))
    sizes.append({"cohort": c, "n": len(tr) + len(va) + len(te)})
sizes = pd.DataFrame(sizes)

big = sizes.nlargest(200, "n").cohort
b = bias[bias.cohort.isin(big)].copy()
# redo FDR over the 200 cohorts only
b["q"] = b.groupby("attribute").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
b["significant"] = b.q <= 0.05
print("n range:", sizes[sizes.cohort.isin(big)].n.min(), "-", sizes.n.max())

attrs = sorted(b.attribute.unique())
ncol = 5
fig, axes = plt.subplots(int(np.ceil(len(attrs) / ncol)), ncol, figsize=(4 * ncol, 3.3 * np.ceil(len(attrs) / ncol)),
                         sharex=True, sharey=True)
for ax, attr in zip(axes.ravel(), attrs):
    d = b[b.attribute == attr]
    ns, sg = d[~d.significant], d[d.significant]
    ax.scatter(ns.detectability, ns.utility, color="#c8c7c3", s=15, alpha=0.9, lw=0, label="not significant")
    ax.scatter(sg.detectability, sg.utility, color="#d6453d", s=25, alpha=0.8, lw=0, label="significant (q < 0.05)")
    ax.set_title(f"{attr}  ({len(sg)}/{len(d)})", fontsize=12)
    ax.set_xlim(0, 1)
    ax.grid(color="#e5e4e0", lw=0.8)
axes.ravel()[0].legend(frameon=False, fontsize=10)
fig.suptitle("BERT: 200 largest cohorts", fontsize=14)
fig.tight_layout()
